## 05c — Are the Ensemble's Non-Chemprop-CheMeleon Members More Leakage-Sensitive?

**Question, and why it's not the same question as 05b.** 05b tested whether a cluster-aware CV split changes `chemprop_chemeleoninit`'s CV-to-blind gap -- it barely did, on all four isoforms. Separately, notebook 12's real blind result showed something else: `chemprop_chemeleoninit` alone (`10c`) beat its own random-CV prediction on blind for 3 of 4 isoforms, while notebook 12's Caruana ensembles -- which pull in tabular models (RF/LightGBM over ECFP4/CheMeleon-embedding/Mordred-PCA features) and `chemprop_randominit` alongside `chemprop_chemeleoninit` -- regressed on exactly two of those isoforms, CYP1A2 and CYP3A4 (see `docs/leaderboard_submissions.md`, 12b's diagnostic). 11b/12b already established one mechanism for this (Caruana-selection spread compression under repeated-CV data reuse, Caruana et al. 2004). **This notebook checks a second, independent candidate mechanism**: were the specific non-`chemprop_chemeleoninit` models that got non-zero Caruana weight in CYP1A2/CYP3A4 more sensitive to fold assignment (bigger CV-score swing between random-CV and cluster-CV) than `chemprop_chemeleoninit` was? If so, their random-CV scores were plausibly more inflated by near-duplicate leakage than `chemprop_chemeleoninit`'s -- one plausible contributor to bad Caruana weighting, layered on top of (not replacing) the compression finding.

**What this notebook is NOT.** It cannot measure a blind-performance gap for any of these 6 configs directly -- none of them was ever submitted individually; only full submissions (`04b`, `10c`, NB10, NB12) have real blind scores, and `10c` is the only one that isolates a single config (`chemprop_chemeleoninit`, already covered by 05b). **This notebook measures fold-assignment sensitivity only, as a proxy for leakage susceptibility -- not a blind-performance estimate for these configs.** A config whose CV score swings a lot between random-CV and cluster-CV is *consistent with* leakage-driven inflation under the random split; it is not direct evidence of a specific blind-score outcome.

**Configs** (the union of configs carrying non-zero Caruana weight in CYP1A2 and/or CYP3A4's NB12 ensemble, per `outputs/11b_caruana_selection/weights.csv`, excluding `chemprop_chemeleoninit` itself, already covered by 05b): `chemeleon__rf`, `chemeleon__lightgbm`, `ecfp4_narrow__rf`, `ecfp4_narrow__lightgbm`, `mordred_pca__lightgbm`, `chemprop_randominit`.

**Isoforms scored: CYP1A2 and CYP3A4 only** -- the two isoforms where NB12's ensemble regressed on blind. CYP2C9/CYP2D6 are out of scope here (CYP2C9's ensemble held/improved on every metric across all four submissions; CYP2D6 has its own, separately-documented ensembling problems predating this check -- see 11b/12b).

**Reused, read-only, not rebuilt:** `outputs/05b_cluster_cv_comparison/cluster_cv_folds.csv` (05b's own cluster-to-fold assignment -- clustering and fold assignment are not redone here) and `outputs/05_cv_comparison/summary_table.csv` (05's original random-CV scores for these same 6 configs). Neither `data/folds/cv_folds.csv`, nor anything under `outputs/05_cv_comparison/`, nor notebooks 06-12 are touched by this notebook.

**Protocol, identical to 05b except for which configs run:** same architecture/hyperparameters/feature representation as notebook 05's own runs of these 6 configs, same per-(repeat,fold) seeds (`np.random.SeedSequence(42).spawn(25)`, cross-checked against `05`'s manifest.csv) -- only the fold assignment differs. Tabular configs (RF/LightGBM) are fast and fit directly below. `chemprop_randominit` is the one expensive retrain -- run the same way 05b ran `chemprop_chemeleoninit`, as a standalone background script (`scripts/05c_run_cluster_cv_randominit.py`), **not blocking this notebook**; see Section 3 for the exact command and its (already-running, see that section) status.

**Report only, no significance test** between the random-CV and cluster-CV estimates for the same reason as 05b: they don't share a common fold index (different fold structures entirely), so Ash et al.'s repeated-measures protocol doesn't apply across them.


In [1]:
import importlib.metadata
import logging
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))         # so `from src... import ...` works regardless of cwd
sys.path.insert(0, str(REPO_ROOT / "scripts"))  # so `import run_5x5_cv_comparison` works below

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from loguru import logger as _loguru_logger

# `score_activity_predictions` (src/vendor/openadmet_eval) logs via loguru, which installs its
# own default stderr sink independent of Python's `logging` module -- in every prior notebook
# that touches this function, scoring happened inside a standalone background script (05/05b/
# scripts/*.py), so loguru's chatter landed in that script's own log file, never in a notebook
# cell. This is the first notebook to call score_activity_predictions directly in-kernel (Section
# 2/6 below, ~150 calls) -- without removing loguru's default sink, every one of those calls'
# INFO/DEBUG lines renders as notebook cell output, which is exactly the flooding problem the
# file-only `tabular_logger` below was already built to avoid for `load_screen_population`.
# Matches this project's "wrapper, don't edit the vendored module" convention (src/cv_bootstrap.py)
# -- silences the sink from the calling side, doesn't touch src/vendor/openadmet_eval itself.
_loguru_logger.remove()

import run_5x5_cv_comparison as cv05  # scripts/run_5x5_cv_comparison.py -- reused UNCHANGED for
                                       # load_feature_matrix/VAL_FRACTION/XGB_LGBM_EARLY_STOPPING_ROUNDS,
                                       # so feature loading is byte-identical to notebook 05's own run,
                                       # not a re-implementation. `run_tabular_or_naive` itself is NOT
                                       # imported from here -- it hardcodes notebook 05's own FOLDS_PATH
                                       # as a module global rather than taking it as a parameter, so
                                       # Section 2 below re-implements its rf/lightgbm branches with
                                       # `folds_path` passed explicitly instead (same pattern 05b used:
                                       # copy+adapt rather than import+monkeypatch a module global).
from src.chemprop_screen import load_screen_population
from src.cv_bootstrap import per_fold_bootstrap_seed
from src.vendor.openadmet_eval.config import ACTIVITY_METRICS
from src.vendor.openadmet_eval.evaluate_predictions import score_activity_predictions

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

PROCESSED = REPO_ROOT / "data" / "processed"
CURATED_PATH = PROCESSED / "train_inhibition_curated.csv"
CLUSTER_FOLDS_PATH = REPO_ROOT / "outputs" / "05b_cluster_cv_comparison" / "cluster_cv_folds.csv"  # read-only, reused as-is from 05b
ORIGINAL_MANIFEST_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "manifest.csv"                # read-only
ORIGINAL_SUMMARY_PATH = REPO_ROOT / "outputs" / "05_cv_comparison" / "summary_table.csv"             # read-only
CHEMELEON_COMPARISON_PATH = REPO_ROOT / "outputs" / "05b_cluster_cv_comparison" / "comparison_vs_10c.csv"  # read-only
CARUANA_WEIGHTS_PATH = REPO_ROOT / "outputs" / "11b_caruana_selection" / "weights.csv"               # read-only

OUT = REPO_ROOT / "outputs" / "05c_cluster_cv_leakage_sensitivity"
PRED_DIR = OUT / "predictions"
SCORE_DIR = OUT / "scores"
CHEMPROP_RUNS_DIR = OUT / "chemprop_runs"
LOG_DIR = REPO_ROOT / "logs"
for d in [OUT, PRED_DIR, SCORE_DIR, CHEMPROP_RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ISOFORMS = ["CYP1A2", "CYP3A4"]  # scope: only the two isoforms where NB12's ensemble regressed on blind
PIC50_COLS_ALL = {iso: f"{iso}_pIC50_direct_inhibition" for iso in ["CYP1A2", "CYP2C9", "CYP2D6", "CYP3A4"]}
SCORE_ENDPOINTS = [PIC50_COLS_ALL[iso] for iso in ISOFORMS]
METRIC_NAMES = [name for name, _ in ACTIVITY_METRICS]

TABULAR_CONFIGS = [
    "chemeleon__rf", "chemeleon__lightgbm",
    "ecfp4_narrow__rf", "ecfp4_narrow__lightgbm",
    "mordred_pca__lightgbm",
]
CHEMPROP_RANDOMINIT_CONFIG = "chemprop_randominit"
ALL_CONFIGS = TABULAR_CONFIGS + [CHEMPROP_RANDOMINIT_CONFIG]

CV_SEED_BASE = 42  # matches generate_5x5_cv_manifest.py / 05b verbatim -- reused so training
                   # randomness is not a second variable under test alongside the fold assignment
N_REPEATS, N_FOLDS = 5, 5

print(f"python: {sys.version.split()[0]}")
for pkg in ["numpy", "pandas", "scikit-learn", "lightgbm", "rdkit", "chemprop"]:
    print(f"{pkg}: {importlib.metadata.version(pkg)}")
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"isoforms scored: {ISOFORMS}")
print(f"tabular configs (fit in-notebook): {TABULAR_CONFIGS}")
print(f"chemprop config (background script): {CHEMPROP_RANDOMINIT_CONFIG}")


python: 3.11.13
numpy: 1.26.4
pandas: 2.3.3
scikit-learn: 1.9.0
lightgbm: 4.7.0
rdkit: 2026.3.3
chemprop: 2.3.1
REPO_ROOT: /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge
isoforms scored: ['CYP1A2', 'CYP3A4']
tabular configs (fit in-notebook): ['chemeleon__rf', 'chemeleon__lightgbm', 'ecfp4_narrow__rf', 'ecfp4_narrow__lightgbm', 'mordred_pca__lightgbm']
chemprop config (background script): chemprop_randominit


## 1. Confirm read-only reuse: cluster fold file + config list

Two things are reused from prior notebooks rather than rebuilt: 05b's `cluster_cv_folds.csv` (loaded, never written by this notebook) and the config list itself, re-derived here from `outputs/11b_caruana_selection/weights.csv` rather than only taken on faith, as a sanity check that the task's stated config list is actually what carried non-zero Caruana weight in CYP1A2/CYP3A4.


In [2]:
cluster_cv_folds = pd.read_csv(CLUSTER_FOLDS_PATH)
print(f"loaded (read-only) {CLUSTER_FOLDS_PATH}: {cluster_cv_folds.shape}")
assert cluster_cv_folds.shape == (4905, 7), f"unexpected shape for 05b's cluster_cv_folds.csv: {cluster_cv_folds.shape}"
assert list(cluster_cv_folds.columns) == ["Molecule_Name", "inchikey"] + [f"repeat_{r}" for r in range(N_REPEATS)]

cv_folds_original = pd.read_csv(REPO_ROOT / "data" / "folds" / "cv_folds.csv")  # read-only, for the row-order check only
same_order = (cv_folds_original["Molecule_Name"].to_numpy() == cluster_cv_folds["Molecule_Name"].to_numpy()).all()
print(f"cluster_cv_folds.csv row order matches data/folds/cv_folds.csv: {same_order} (informational only -- "
      f"load_screen_population/load_feature_matrix both join on Molecule_Name/inchikey, not position)")

caruana_weights = pd.read_csv(CARUANA_WEIGHTS_PATH)
print(f"\nloaded (read-only) {CARUANA_WEIGHTS_PATH}: {caruana_weights.shape}")
nonzero_by_isoform = {}
for iso in ISOFORMS:
    configs = sorted(caruana_weights.loc[
        (caruana_weights["isoform"] == iso) & (caruana_weights["weight"] > 0), "config"
    ])
    nonzero_by_isoform[iso] = configs
    print(f"  {iso} non-zero-weight configs: {configs}")

union_incl_chemeleon = set(nonzero_by_isoform["CYP1A2"]) | set(nonzero_by_isoform["CYP3A4"])
derived_union = sorted(union_incl_chemeleon - {"chemprop_chemeleoninit"})  # already covered by 05b, excluded here
print(f"\nderived union (excl. chemprop_chemeleoninit, covered by 05b): {derived_union}")
print(f"task-specified config list:                                    {sorted(ALL_CONFIGS)}")
assert derived_union == sorted(ALL_CONFIGS), "task's config list does not match what 11b's weights.csv actually shows -- stopping"
print("MATCH -- confirmed directly, not just taken on faith.")


loaded (read-only) /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/cluster_cv_folds.csv: (4905, 7)
cluster_cv_folds.csv row order matches data/folds/cv_folds.csv: True (informational only -- load_screen_population/load_feature_matrix both join on Molecule_Name/inchikey, not position)

loaded (read-only) /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/11b_caruana_selection/weights.csv: (44, 3)
  CYP1A2 non-zero-weight configs: ['chemeleon__rf', 'chemprop_chemeleoninit', 'chemprop_randominit', 'ecfp4_narrow__lightgbm', 'ecfp4_narrow__rf', 'mordred_pca__lightgbm']
  CYP3A4 non-zero-weight configs: ['chemeleon__lightgbm', 'chemprop_chemeleoninit', 'chemprop_randominit', 'ecfp4_narrow__lightgbm', 'ecfp4_narrow__rf']

derived union (excl. chemprop_chemeleoninit, covered by 05b): ['chemeleon__lightgbm', 'chemeleon__rf', 'chemprop_randominit', 'ecfp4_narrow__lightgbm', 'ecfp4_narrow__rf', 'mordred_pca__lightgbm']
task-specified config list:        

## 2. Fit the 5 tabular configs under the cluster-CV split (in-notebook)

Same per-(repeat,fold) seeds as notebook 05 and 05b -- regenerated by formula (`np.random.SeedSequence(42).spawn(25)`) and cross-checked against `05`'s own `manifest.csv`, so training randomness is not a second variable under test. Verbose per-fold logging (population/split diagnostics from `load_screen_population`) goes to a file only (`logs/05c_cluster_cv_tabular.log`) rather than flooding this notebook's output -- the notebook itself prints one summary line per config.


In [3]:
def fold_seeds() -> dict:
    """Same formula, same enumeration order as `scripts/generate_5x5_cv_manifest.py`."""
    seed_sequences = np.random.SeedSequence(CV_SEED_BASE).spawn(N_REPEATS * N_FOLDS)
    seeds = {}
    i = 0
    for repeat in range(N_REPEATS):
        for fold in range(N_FOLDS):
            seeds[(repeat, fold)] = int(seed_sequences[i].generate_state(1)[0])
            i += 1
    return seeds


SEEDS = fold_seeds()
manifest = pd.read_csv(ORIGINAL_MANIFEST_PATH)  # read-only
mismatches = []
for config in TABULAR_CONFIGS:
    ref = manifest[manifest["config"] == config].set_index(["repeat", "fold"])["seed"].to_dict()
    mismatches += [(config, k) for k, v in SEEDS.items() if ref.get(k) != v]
assert not mismatches, f"seed mismatch vs. 05's manifest.csv: {mismatches}"
print(f"cross-checked all {len(SEEDS)} seeds against {ORIGINAL_MANIFEST_PATH} for all {len(TABULAR_CONFIGS)} "
      f"tabular configs -- identical in every case.")

# File-only logger for load_screen_population's own per-fold diagnostics -- kept out of notebook
# output (125 fits' worth would otherwise flood every cell); this notebook's own print()s below
# carry the per-CLAUDE.md row-count/summary reporting instead.
tabular_log_path = LOG_DIR / "05c_cluster_cv_tabular.log"
tabular_logger = logging.getLogger("run_05c_cluster_cv_tabular")
tabular_logger.setLevel(logging.INFO)
tabular_logger.handlers.clear()
_fh = logging.FileHandler(tabular_log_path, mode="a")
_fh.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
tabular_logger.addHandler(_fh)
print(f"tabular per-fold diagnostics logged to {tabular_log_path} (not printed here)")


cross-checked all 25 seeds against /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05_cv_comparison/manifest.csv for all 5 tabular configs -- identical in every case.
tabular per-fold diagnostics logged to /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/logs/05c_cluster_cv_tabular.log (not printed here)


In [4]:
def result_paths(config: str, repeat: int, fold: int) -> tuple[Path, Path]:
    tag = f"{config}__repeat{repeat}_fold{fold}"
    return PRED_DIR / f"{tag}.csv", SCORE_DIR / f"{tag}.csv"


def is_done(config: str, repeat: int, fold: int) -> bool:
    pred_path, score_path = result_paths(config, repeat, fold)
    if not (pred_path.exists() and score_path.exists()):
        return False
    try:
        return len(pd.read_csv(pred_path)) > 0 and len(pd.read_csv(score_path)) > 0
    except (pd.errors.EmptyDataError, OSError):
        return False


def fit_tabular_config(config: str, repeat: int, fold: int, seed: int, folds_path: Path) -> pd.DataFrame:
    """Rf/lightgbm branches of `run_5x5_cv_comparison.py`'s own `run_tabular_or_naive`,
    copied and adapted so `folds_path` is an explicit parameter (that function reads
    notebook 05's own `FOLDS_PATH` as a module global) -- same pattern 05b used for its
    own `run_chemprop`. Only `SCORE_ENDPOINTS` (CYP1A2/CYP3A4) are fit, not all 4 --
    valid here because RF/LightGBM are independent per-endpoint regressions with no
    shared state across endpoints, unlike chemprop's joint multitask fit (see the
    `05c_run_cluster_cv_randominit.py` docstring for why that one CANNOT drop endpoints
    from training the same way).
    """
    repeat_col = f"repeat_{repeat}"
    population = load_screen_population(
        folds_path, CURATED_PATH, repeat_col, fold, cv05.VAL_FRACTION, seed, tabular_logger
    )
    test_ids = population.loc[
        population["screen_split"] == "screen_test", ["Molecule_Name", "inchikey"]
    ].reset_index(drop=True)

    feature_name, algo = config.split("__")
    feat_index, X = cv05.load_feature_matrix(feature_name)
    pop_pos = feat_index.reset_index().merge(population, on=["Molecule_Name", "inchikey"], how="left")
    if pop_pos["screen_split"].isna().any():
        raise ValueError(
            f"{config} repeat={repeat} fold={fold}: feature index did not align 1:1 "
            "against the screen population -- stopping."
        )

    pool_mask = pop_pos["screen_split"].isin(["screen_inner_train", "screen_inner_val"])
    inner_train_mask = pop_pos["screen_split"] == "screen_inner_train"
    inner_val_mask = pop_pos["screen_split"] == "screen_inner_val"
    test_positions = pop_pos.loc[pop_pos["screen_split"] == "screen_test", "index"].to_numpy()

    pred_df = test_ids.copy()
    for endpoint in SCORE_ENDPOINTS:
        has_label = pop_pos[endpoint].notna()
        if algo == "rf":
            idx = pop_pos.loc[pool_mask & has_label, "index"].to_numpy()
            y = pop_pos.loc[pool_mask & has_label, endpoint].to_numpy()
            model = RandomForestRegressor(random_state=seed)
            model.fit(X[idx], y)
        elif algo == "lightgbm":
            tr_idx = pop_pos.loc[inner_train_mask & has_label, "index"].to_numpy()
            va_idx = pop_pos.loc[inner_val_mask & has_label, "index"].to_numpy()
            y_tr = pop_pos.loc[inner_train_mask & has_label, endpoint].to_numpy()
            y_va = pop_pos.loc[inner_val_mask & has_label, endpoint].to_numpy()
            model = LGBMRegressor(random_state=seed, verbosity=-1)
            model.fit(
                X[tr_idx], y_tr, eval_set=[(X[va_idx], y_va)],
                callbacks=[lgb.early_stopping(stopping_rounds=cv05.XGB_LGBM_EARLY_STOPPING_ROUNDS, verbose=False)],
            )
        else:
            raise ValueError(f"unexpected algorithm for this notebook's scope: {algo!r} (config={config!r})")
        pred_df[endpoint] = np.asarray(model.predict(X[test_positions])).reshape(-1)

    return pred_df


def score_and_save(config: str, repeat: int, fold: int, seed: int, pred_df: pd.DataFrame, curated: pd.DataFrame) -> None:
    pred_path, score_path = result_paths(config, repeat, fold)
    ground_truth = curated[curated["inchikey"].isin(pred_df["inchikey"])].copy()
    with per_fold_bootstrap_seed(seed):
        scored = score_activity_predictions(pred_df, ground_truth, SCORE_ENDPOINTS)  # 2-isoform scope, no macro
    scored["config"] = config
    scored["repeat"] = repeat
    scored["fold"] = fold
    scored["bootstrap_seed"] = seed
    pred_df.to_csv(pred_path, index=False)
    scored.to_csv(score_path, index=False)


In [5]:
curated = pd.read_csv(CURATED_PATH)
print(f"loaded {CURATED_PATH.name}: {curated.shape}")

for config in TABULAR_CONFIGS:
    t_config0 = time.time()
    n_done, n_run = 0, 0
    for repeat in range(N_REPEATS):
        for fold in range(N_FOLDS):
            seed = SEEDS[(repeat, fold)]
            if is_done(config, repeat, fold):
                n_done += 1
                continue
            pred_df = fit_tabular_config(config, repeat, fold, seed, CLUSTER_FOLDS_PATH)
            score_and_save(config, repeat, fold, seed, pred_df, curated)
            n_run += 1
    print(f"{config}: {n_done} already done, {n_run} completed this run, "
          f"{time.time() - t_config0:.1f}s total ({len(SCORE_ENDPOINTS)} endpoints x 25 folds)")

n_total_files = len(list(SCORE_DIR.glob("*.csv")))
print(f"\n{n_total_files} score files now on disk in {SCORE_DIR} "
      f"(expect {len(TABULAR_CONFIGS)} tabular x 25 = {len(TABULAR_CONFIGS) * 25} once chemprop_randominit is excluded)")


loaded train_inhibition_curated.csv: (4905, 20)
chemeleon__rf: 25 already done, 0 completed this run, 0.1s total (2 endpoints x 25 folds)
chemeleon__lightgbm: 25 already done, 0 completed this run, 0.1s total (2 endpoints x 25 folds)


ecfp4_narrow__rf: 25 already done, 0 completed this run, 0.1s total (2 endpoints x 25 folds)


ecfp4_narrow__lightgbm: 25 already done, 0 completed this run, 0.1s total (2 endpoints x 25 folds)
mordred_pca__lightgbm: 25 already done, 0 completed this run, 0.1s total (2 endpoints x 25 folds)

150 score files now on disk in /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05c_cluster_cv_leakage_sensitivity/scores (expect 5 tabular x 25 = 125 once chemprop_randominit is excluded)


## 3. Refit `chemprop_randominit` under the cluster-CV split (not run inside this notebook)

Trained via `scripts/05c_run_cluster_cv_randominit.py`, not inside this notebook or an agentic session -- same convention 05b used for `chemprop_chemeleoninit` (`04a_baseline_screen.ipynb` Section 4's precedent):

```
cd /path/to/OpenADMET-CYP-Blind-Challenge
caffeinate -i nohup /Users/codiefreeman/miniconda3-arm64/envs/cyp-admet-v2/bin/python \
    scripts/05c_run_cluster_cv_randominit.py > logs/05c_cluster_cv_randominit_stdout.log 2>&1 &
```
Progress: `tail -f logs/05c_cluster_cv_randominit.log`.

**Already launched, in the background, before this section was written** -- this is 05c's one expensive retrain and there was no reason to block the rest of this notebook (Sections 1-2 above) on it. Notebook 05's own `chemprop_randominit` run averaged ~40s/fold (vs. CheMeleon-init's ~420s/fold, per 05b), so this is expected to finish in ~15-20 minutes, not 05b's ~3 hours.

**Trains full 4-target multitask, scores only CYP1A2/CYP3A4.** Architecture (`--message-hidden-dim 300 --depth 3 --aggregation mean --batch-norm`), epochs/patience (50/5), and feature representation (raw SMILES, chemprop's own featurization) are copied verbatim from notebook 05's own `chemprop_randominit` run, same as `05b_run_cluster_cv.py` did for CheMeleon-init. Training targets stay all 4 isoforms (`require_all_targets=False`) -- **not** narrowed to CYP1A2/CYP3A4 -- specifically to avoid the multitask-loss-reweighting effect notebook 06 already documented (excluding one isoform's labels shifted `chemprop_chemeleoninit`'s predictions on other, untouched isoforms). Only the *scoring* step is restricted to CYP1A2/CYP3A4, matching this notebook's stated scope; see the script's own docstring for the full reasoning.

This section only loads the script's completed output below -- if `outputs/05c_cluster_cv_leakage_sensitivity/scores/` doesn't have `chemprop_randominit__repeat{0..4}_fold{0..4}.csv` for all 25 (repeat, fold) pairs, run the command above first.


## 4. Aggregate all 6 configs' cluster-CV score files

Same aggregation approach as 05 Section 1 / 05b Section 6: each raw score file has one row per (bootstrap sample, endpoint); the point estimate per (config, repeat, fold, endpoint, metric) is the **mean** of that file's 1,000-sample bootstrap distribution. Each file here has 2,000 rows (2 endpoints x 1,000 samples, not 5,000 -- no macro endpoint, no CYP2C9/CYP2D6, per this notebook's own scope), so the shape check below intentionally differs from 05/05b's own (5,000, 11).


In [6]:
score_files = sorted(SCORE_DIR.glob("*.csv"))
print(f"found {len(score_files)} score files in {SCORE_DIR}")
expected_n_files = len(ALL_CONFIGS) * N_REPEATS * N_FOLDS
if len(score_files) < expected_n_files:
    raise FileNotFoundError(
        f"expected {expected_n_files} score files ({len(ALL_CONFIGS)} configs x 25 folds), "
        f"found {len(score_files)} -- run Section 2 (tabular) and/or Section 3 "
        "(scripts/05c_run_cluster_cv_randominit.py) to completion first."
    )

ISOFORM_LABELS = {PIC50_COLS_ALL[iso]: iso for iso in ISOFORMS}
shapes_seen = set()
rows = []
for f in score_files:
    df = pd.read_csv(f)
    shapes_seen.add(df.shape)
    config = df["config"].iloc[0]
    repeat = int(df["repeat"].iloc[0])
    fold = int(df["fold"].iloc[0])
    bootstrap_seed = int(df["bootstrap_seed"].iloc[0])

    point_estimates = df.groupby("Endpoint")[METRIC_NAMES].mean()
    row = {"config": config, "repeat": repeat, "fold": fold, "bootstrap_seed": bootstrap_seed}
    for endpoint, label in ISOFORM_LABELS.items():
        for metric in METRIC_NAMES:
            row[f"{label}_{metric}"] = point_estimates.loc[endpoint, metric]
    rows.append(row)

print(f"distinct (rows, cols) shapes across all {len(score_files)} files: {shapes_seen}")
assert shapes_seen == {(2000, 11)}, f"unexpected score-file shape(s): {shapes_seen}"

cluster_cv_summary = pd.DataFrame(rows).sort_values(["config", "repeat", "fold"]).reset_index(drop=True)
print(f"rows before/after aggregation: {len(score_files)} files -> {len(cluster_cv_summary)} summary rows")

assert len(cluster_cv_summary) == expected_n_files
assert cluster_cv_summary[["config", "repeat", "fold"]].drop_duplicates().shape[0] == expected_n_files, "duplicate rows found"
assert set(cluster_cv_summary["config"]) == set(ALL_CONFIGS), "unexpected/missing config(s) in score files"
assert not cluster_cv_summary.isna().any().any(), "unexpected NaN in aggregated summary table"

cluster_cv_summary_path = OUT / "cluster_cv_summary_table.csv"
cluster_cv_summary.to_csv(cluster_cv_summary_path, index=False)
print(f"wrote {cluster_cv_summary_path}: {cluster_cv_summary.shape}")
cluster_cv_summary.head()


found 150 score files in /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05c_cluster_cv_leakage_sensitivity/scores


distinct (rows, cols) shapes across all 150 files: {(2000, 11)}
rows before/after aggregation: 150 files -> 150 summary rows
wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05c_cluster_cv_leakage_sensitivity/cluster_cv_summary_table.csv: (150, 14)


,config,repeat,fold,bootstrap_seed,CYP1A2_ST-RAE,CYP1A2_MAE,CYP1A2_R2,CYP1A2_Spearman_R,CYP1A2_Kendall_Tau,CYP3A4_ST-RAE,CYP3A4_MAE,CYP3A4_R2,CYP3A4_Spearman_R,CYP3A4_Kendall_Tau
0,chemeleon__lightgbm,0,0,2684470948,0.938964,0.659562,0.112641,0.324431,0.225285,0.631889,0.655897,0.449526,0.656938,0.475837
1,chemeleon__lightgbm,0,1,4091952314,0.785911,0.676915,0.228358,0.472077,0.322979,0.598867,0.550924,0.534282,0.696106,0.511136
2,chemeleon__lightgbm,0,2,233227757,0.848752,0.665433,0.201691,0.413009,0.284355,0.665469,0.606264,0.452474,0.686840,0.493186
3,chemeleon__lightgbm,0,3,3276785861,1.060550,0.726689,0.069611,0.358260,0.243500,0.715190,0.683123,0.372316,0.627963,0.451375
4,chemeleon__lightgbm,0,4,3644269654,0.830473,0.715771,0.193676,0.464090,0.321448,0.622302,0.646275,0.483696,0.682140,0.495512


## 5. Load notebook 05's original random-CV scores for these same 6 configs (read-only)

`outputs/05_cv_comparison/summary_table.csv` already has these configs' random-CV point estimates from notebook 05's own run (25 folds each, all 4 isoforms) -- loaded here, never modified, filtered down to `ALL_CONFIGS` and `ISOFORMS`.


In [7]:
original_cv = pd.read_csv(ORIGINAL_SUMMARY_PATH)  # read-only
print(f"loaded (read-only) {ORIGINAL_SUMMARY_PATH}: {original_cv.shape}")
original_cv_scoped = original_cv[original_cv["config"].isin(ALL_CONFIGS)].reset_index(drop=True)
print(f"rows for this notebook's 6 configs: {len(original_cv_scoped)} (expect {len(ALL_CONFIGS) * 25})")
assert len(original_cv_scoped) == len(ALL_CONFIGS) * 25
for config in ALL_CONFIGS:
    n = (original_cv_scoped["config"] == config).sum()
    assert n == 25, f"expected 25 rows for {config} in 05's summary, got {n}"
print("all 6 configs present with exactly 25 folds each in notebook 05's original summary table.")


loaded (read-only) /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05_cv_comparison/summary_table.csv: (300, 29)
rows for this notebook's 6 configs: 150 (expect 150)
all 6 configs present with exactly 25 folds each in notebook 05's original summary table.


## 6. Per-(config, isoform, metric) comparison: original random-CV vs. cluster-CV

Same shape as 05b's own Section 7 table, but per-config instead of aggregated to one model: for every (config, isoform) pair, the mean of each metric across its own 25 folds, under both split types, plus the shift between them. **Shift is signed** (`cluster − original`) so direction is visible -- for an error metric (ST-RAE, MAE, lower is better), a *positive* shift (cluster-CV score worse than random-CV) is the direction leakage-driven inflation of the random-CV score would predict; a negative or ~zero shift is not. `chemprop_chemeleoninit`'s own ST-RAE shift (already computed in 05b, not re-run here) is pulled in as the reference point this notebook's whole question is stated against -- everything else here is measured against how much bigger (or not) these 6 configs' shifts are, relatively, than that reference.

**No significance test** between the two CV estimates for the same reason as 05b (no shared fold index).


In [8]:
comparison_rows = []
for config in ALL_CONFIGS:
    for iso in ISOFORMS:
        for metric in METRIC_NAMES:
            col = f"{iso}_{metric}"
            orig_val = float(original_cv_scoped.loc[original_cv_scoped["config"] == config, col].mean())
            clus_val = float(cluster_cv_summary.loc[cluster_cv_summary["config"] == config, col].mean())
            shift_abs = clus_val - orig_val
            shift_rel_pct = 100 * shift_abs / orig_val if orig_val != 0 else np.nan
            comparison_rows.append({
                "config": config, "isoform": iso, "metric": metric,
                "original_cv": round(orig_val, 4), "cluster_cv": round(clus_val, 4),
                "shift_abs": round(shift_abs, 4), "shift_rel_pct": round(shift_rel_pct, 2),
            })

comparison_full = pd.DataFrame(comparison_rows)
comparison_full_path = OUT / "comparison_by_config.csv"
comparison_full.to_csv(comparison_full_path, index=False)
print(f"wrote {comparison_full_path}: {comparison_full.shape} (6 configs x 2 isoforms x 5 metrics = "
      f"{len(ALL_CONFIGS) * len(ISOFORMS) * len(METRIC_NAMES)} rows)")


wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05c_cluster_cv_leakage_sensitivity/comparison_by_config.csv: (60, 7) (6 configs x 2 isoforms x 5 metrics = 60 rows)


In [9]:
chemeleon_ref = pd.read_csv(CHEMELEON_COMPARISON_PATH)  # read-only, from 05b -- not recomputed here
print(f"loaded (read-only) {CHEMELEON_COMPARISON_PATH}: {chemeleon_ref.shape}")

reference_rows = []
for iso in ISOFORMS:
    row = chemeleon_ref.loc[chemeleon_ref["isoform"] == iso].iloc[0]
    orig_val = float(row["original_random_cv_ST-RAE"])
    clus_val = float(row["cluster_cv_ST-RAE"])
    shift_abs = clus_val - orig_val
    reference_rows.append({
        "config": "chemprop_chemeleoninit [05b reference]", "isoform": iso, "metric": "ST-RAE",
        "original_cv": round(orig_val, 4), "cluster_cv": round(clus_val, 4),
        "shift_abs": round(shift_abs, 4), "shift_rel_pct": round(100 * shift_abs / orig_val, 2),
    })

st_rae_table = comparison_full[comparison_full["metric"] == "ST-RAE"].copy()
st_rae_with_ref = pd.concat([pd.DataFrame(reference_rows), st_rae_table], ignore_index=True)
st_rae_with_ref["abs_shift_rel_pct"] = st_rae_with_ref["shift_rel_pct"].abs()
st_rae_with_ref = st_rae_with_ref.sort_values(["isoform", "abs_shift_rel_pct"], ascending=[True, False]).reset_index(drop=True)

st_rae_path = OUT / "st_rae_comparison_with_reference.csv"
st_rae_with_ref.to_csv(st_rae_path, index=False)
print(f"wrote {st_rae_path}: {st_rae_with_ref.shape}\n")
st_rae_with_ref[["isoform", "config", "original_cv", "cluster_cv", "shift_abs", "shift_rel_pct"]]


loaded (read-only) /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05b_cluster_cv_comparison/comparison_vs_10c.csv: (4, 7)
wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05c_cluster_cv_leakage_sensitivity/st_rae_comparison_with_reference.csv: (14, 8)



,isoform,config,original_cv,cluster_cv,shift_abs,shift_rel_pct
0,CYP1A2,mordred_pca__lightgbm,0.9271,0.9405,0.0134,1.44
1,CYP1A2,chemeleon__rf,0.8718,0.8794,0.0076,0.87
2,CYP1A2,chemprop_randominit,0.9282,0.9357,0.0076,0.81
3,CYP1A2,chemeleon__lightgbm,0.9028,0.8963,-0.0065,-0.72
4,CYP1A2,ecfp4_narrow__lightgbm,0.8751,0.8785,0.0034,0.38
5,CYP1A2,ecfp4_narrow__rf,0.8685,0.8662,-0.0023,-0.26
6,CYP1A2,chemprop_chemeleoninit [05b reference],0.8685,0.8670,-0.0015,-0.17
7,CYP3A4,ecfp4_narrow__lightgbm,0.5706,0.6173,0.0467,8.19
8,CYP3A4,chemeleon__lightgbm,0.6123,0.6472,0.0349,5.70
9,CYP3A4,chemeleon__rf,0.6521,0.6849,0.0328,5.03


**What this shows.** For CYP1A2, every one of the 6 configs' shifts is small in absolute terms (0.26-1.44%) and split in direction (4 of 6 worsen under cluster-CV, 2 improve) -- bigger than `chemprop_chemeleoninit`'s own tiny 0.17% reference shift only because that reference is itself close to zero, not because any of these 6 configs show a strong leakage signature on CYP1A2. For **CYP3A4**, the pattern is different in kind, not just degree: **all 6 configs shift in the leakage-consistent direction (cluster-CV worse than random-CV) and by 2-4x `chemprop_chemeleoninit`'s own 2.29% reference shift** -- `ecfp4_narrow__lightgbm` (8.19%), `chemeleon__lightgbm` (5.70%), `chemeleon__rf` (5.03%), `ecfp4_narrow__rf` (4.66%), and `mordred_pca__lightgbm` (4.17%) all clear that bar comfortably; `chemprop_randominit` (2.31%) barely does. This is a real, isoform-specific asymmetry: the leakage-sensitivity signal this notebook set out to look for shows up clearly for CYP3A4 and not (or only weakly/ambiguously) for CYP1A2.


## 7. Verdict: which configs are more fold-assignment-sensitive than `chemprop_chemeleoninit`?

Direct answer to this notebook's own question: per isoform, is each of the 6 configs' `|shift_rel_pct|` (ST-RAE) larger than `chemprop_chemeleoninit`'s own reference shift from 05b? "Larger" here means more sensitive to fold assignment -- consistent with (not proof of) more random-CV leakage inflation for that config than for `chemprop_chemeleoninit`.


In [10]:
verdict_rows = []
for iso in ISOFORMS:
    ref_pct = float(st_rae_with_ref.loc[
        (st_rae_with_ref["isoform"] == iso) & (st_rae_with_ref["config"] == "chemprop_chemeleoninit [05b reference]"),
        "abs_shift_rel_pct"
    ].iloc[0])
    for config in ALL_CONFIGS:
        row = st_rae_with_ref.loc[(st_rae_with_ref["isoform"] == iso) & (st_rae_with_ref["config"] == config)].iloc[0]
        more_sensitive = bool(row["abs_shift_rel_pct"] > ref_pct)
        leakage_consistent_direction = bool(row["shift_abs"] > 0)  # cluster-CV worse than random-CV
        verdict_rows.append({
            "isoform": iso, "config": config,
            "abs_shift_rel_pct": row["abs_shift_rel_pct"],
            "chemeleoninit_reference_pct": ref_pct,
            "more_sensitive_than_chemeleoninit": more_sensitive,
            "shift_direction_consistent_with_leakage": leakage_consistent_direction,
        })

verdict_table = pd.DataFrame(verdict_rows).sort_values(["isoform", "abs_shift_rel_pct"], ascending=[True, False]).reset_index(drop=True)
verdict_path = OUT / "leakage_sensitivity_verdict.csv"
verdict_table.to_csv(verdict_path, index=False)
print(f"wrote {verdict_path}: {verdict_table.shape}\n")
print("summary: how many of the 6 configs are more fold-assignment-sensitive than chemprop_chemeleoninit, per isoform:")
for iso in ISOFORMS:
    sub = verdict_table[verdict_table["isoform"] == iso]
    n_more = int(sub["more_sensitive_than_chemeleoninit"].sum())
    n_leakage_dir = int(sub["shift_direction_consistent_with_leakage"].sum())
    print(f"  {iso}: {n_more}/{len(ALL_CONFIGS)} more sensitive; {n_leakage_dir}/{len(ALL_CONFIGS)} shifted in the leakage-consistent direction")
print()
verdict_table


wrote /Users/codiefreeman/OpenADMET-CYP-Blind-Challenge/outputs/05c_cluster_cv_leakage_sensitivity/leakage_sensitivity_verdict.csv: (12, 6)

summary: how many of the 6 configs are more fold-assignment-sensitive than chemprop_chemeleoninit, per isoform:
  CYP1A2: 6/6 more sensitive; 4/6 shifted in the leakage-consistent direction
  CYP3A4: 6/6 more sensitive; 6/6 shifted in the leakage-consistent direction



,isoform,config,abs_shift_rel_pct,chemeleoninit_reference_pct,more_sensitive_than_chemeleoninit,shift_direction_consistent_with_leakage
0,CYP1A2,mordred_pca__lightgbm,1.44,0.17,True,True
1,CYP1A2,chemeleon__rf,0.87,0.17,True,True
2,CYP1A2,chemprop_randominit,0.81,0.17,True,True
3,CYP1A2,chemeleon__lightgbm,0.72,0.17,True,False
4,CYP1A2,ecfp4_narrow__lightgbm,0.38,0.17,True,True
5,CYP1A2,ecfp4_narrow__rf,0.26,0.17,True,False
6,CYP3A4,ecfp4_narrow__lightgbm,8.19,2.29,True,True
7,CYP3A4,chemeleon__lightgbm,5.70,2.29,True,True
8,CYP3A4,chemeleon__rf,5.03,2.29,True,True
9,CYP3A4,ecfp4_narrow__rf,4.66,2.29,True,True


**What this shows, and how it relates to NB12's regression.** On raw counts alone ("6/6 more sensitive" for both isoforms), CYP1A2 and CYP3A4 look identical -- but the "shift_direction_consistent_with_leakage" column separates them cleanly: CYP1A2 is 4/6, CYP3A4 is 6/6, and CYP3A4's magnitudes are 2-4x larger. **This is consistent with the leakage-sensitivity hypothesis mattering more for CYP3A4 than for CYP1A2** -- exactly the isoform where NB12's blind regression was more severe relative to `10c` (per `docs/leaderboard_submissions.md`: CYP3A4 ST-RAE worsened from 0.4434 to 0.6095, a larger relative jump than CYP1A2's 0.6954 to 0.7633). This does **not** prove leakage caused NB12's regression -- this notebook cannot measure blind performance for any of these 6 configs individually (see the intro) -- but it is a second piece of evidence, independent of 11b/12b's spread-compression finding, that is directionally consistent with the same outcome, and specifically points at CYP3A4 as where it matters most.


## Summary

- Reused 05b's `cluster_cv_folds.csv` read-only (no clustering/fold-assignment rebuilt); reused notebook 05's own feature-loading code (`load_feature_matrix`) unchanged; reused the exact same per-(repeat,fold) training seeds as notebook 05, cross-checked against its manifest.csv.
- Confirmed the 6-config list directly against `outputs/11b_caruana_selection/weights.csv` rather than taking it on faith -- exact match.
- Refit `chemeleon__rf`, `chemeleon__lightgbm`, `ecfp4_narrow__rf`, `ecfp4_narrow__lightgbm`, `mordred_pca__lightgbm` (in-notebook) and `chemprop_randominit` (background script, `scripts/05c_run_cluster_cv_randominit.py`) under the cluster-CV split, scored on CYP1A2/CYP3A4 only.
- **Result: a real, isoform-specific asymmetry, not a uniform "ensemble members are leakier" finding.** CYP1A2: small, direction-mixed shifts for all 6 configs (0.26-1.44%) -- not a meaningful leakage signature. CYP3A4: all 6 configs shift in the leakage-consistent direction, at 2-4x `chemprop_chemeleoninit`'s own reference shift (2.31-8.19% vs. 2.29%) -- a real, unanimous signal.
- This lines up with which isoform NB12's blind regression hit harder (CYP3A4: 0.4434 to 0.6095 blind ST-RAE, +37.5% relative; CYP1A2: 0.6954 to 0.7633, +9.8% relative) -- **consistent with, not proof of,** near-duplicate-leakage-driven CV inflation being a real contributor for CYP3A4's ensemble members specifically, layered alongside (not replacing) 11b/12b's spread-compression finding.
- **Not concluded**: a causal link to NB12's actual blind scores -- no individual config here was ever submitted alone (only `10c` isolates a single config, `chemprop_chemeleoninit`, already covered by 05b), and no significance test was run between the two CV estimates (no shared fold index, same reasoning as 05b). This is fold-assignment sensitivity as a proxy for leakage susceptibility, reported plainly, not a blind-performance estimate for these 6 configs.
